In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, precision_recall_curve
from scipy.sparse import hstack, csr_matrix

In [ ]:
fb_pii_df = pd.read_csv("FC pii_detection_results_with_regex_and_gazetteer.csv", delimiter=',', encoding='utf-8-sig', engine='python')

In [ ]:
fb_pii_df.head(5)

,Original_DataFrame_Index,Original_Text,PII_Entity,PII_Type,Score,Source,Exposure_Score
0,0,I hope it’s okay to gently share something her...,reading,B-LOC,1.000000,Gazetteer,0.0
1,1,The Hanover Company is NOW HIRING! Position: ...,hanover,B-STATE,0.981911,Model,0.0
2,1,The Hanover Company is NOW HIRING! Position: ...,hanover,B-STATE,0.966915,Model,0.0
3,1,The Hanover Company is NOW HIRING! Position: ...,+,B-PHONENUMBER,0.925023,Model,0.0
4,1,The Hanover Company is NOW HIRING! Position: ...,1,I-PHONENUMBER,0.995683,Model,0.0


In [ ]:
# 1. טעינת הנתונים והסרת שורות ריקות
file_path = "FC pii_detection_results_with_regex_and_gazetteer.csv"
df = pd.read_csv(file_path).dropna(subset=['Original_DataFrame_Index'])

In [ ]:
# 2. יצירת עמודת מטרה (Target)
# קיבוץ לפי פוסט: אם קיימת לפחות ישות אחת של PII_Entity, הערך יהיה 1. אחרת 0.
df_grouped = df.groupby(['Original_DataFrame_Index', 'Original_Text'], as_index=False).agg(
    has_pii=('PII_Entity', lambda x: x.notna().any())
)
df_grouped['Target'] = df_grouped['has_pii'].astype(int)

In [ ]:
# 3. הנדסת מאפייני מטא-דאטה
def get_meta_features(text):
    text = str(text)
    length = len(text)
    word_count = len(text.split())

    if length == 0:
        return [0, 0, 0, 0]

    upper_ratio = sum(1 for c in text if c.isupper()) / length
    digit_count = sum(1 for c in text if c.isdigit())

    return [length, word_count, upper_ratio, digit_count]

# יצירת מטריצת מטא-דאטה
meta_features = np.array([get_meta_features(t) for t in df_grouped['Original_Text']])

In [ ]:
# 4. עיבוד מקדים לטקסט
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text) # הסרת קישורים
    text = re.sub(r'[^a-zA-Z\s]', '', text) # הסרת תווים מיוחדים ומספרים (נשאיר אותיות בלבד)
    text = text.lower() # אותיות קטנות
    return text

df_grouped['Clean_Text'] = df_grouped['Original_Text'].apply(clean_text)

In [ ]:
# 5. פיצול הנתונים לאימון ובדיקה (Train-Test Split with Stratify)
X_text = df_grouped['Clean_Text']
X_meta = meta_features
y = df_grouped['Target']

X_text_train, X_text_test, X_meta_train, X_meta_test, y_train, y_test = train_test_split(
    X_text, X_meta, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 6. TF-IDF
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=2000)
X_text_train_tfidf = tfidf.fit_transform(X_text_train)
X_text_test_tfidf = tfidf.transform(X_text_test)

In [ ]:
# 7. Scaling (נורמליזציה רק למטא-דאטה)
scaler = StandardScaler()
X_meta_train_scaled = scaler.fit_transform(X_meta_train)
X_meta_test_scaled = scaler.transform(X_meta_test)

In [ ]:
# 8. חיבור מאפיינים: וקטורי TF-IDF ומטא-דאטה
X_train_final = hstack([X_text_train_tfidf, csr_matrix(X_meta_train_scaled)])
X_test_final = hstack([X_text_test_tfidf, csr_matrix(X_meta_test_scaled)])

In [ ]:
# 9. הגדרת המודל וה-GridSearchCV לאופטימיזציית F1-Score
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1
)

print("מתחיל אימון ואופטימיזציה (Grid Search)...")
grid_search.fit(X_train_final, y_train)
best_model = grid_search.best_estimator_
print("היפר-פרמטרים אופטימליים שנבחרו:", grid_search.best_params_)

מתחיל אימון ואופטימיזציה (Grid Search)...
היפר-פרמטרים אופטימליים שנבחרו: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 300}


In [ ]:
# 10. Threshold Tuning למקסום F1 (על סט הבדיקה לצורך הדגמה, בפרקטיקה נהוג לעשות על סט Validation)
y_pred_probs = best_model.predict_proba(X_test_final)[:, 1]

# מציאת Threshold אופטימלי
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_probs)
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10) # 1e-10 למניעת חלוקה באפס
best_thresh_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_thresh_idx] if best_thresh_idx < len(thresholds) else 0.5

print(f"סף אופטימלי (Threshold) למקסום F1: {best_threshold:.4f}")

# חיזוי סופי באמצעות ה-Threshold האופטימלי
y_pred_tuned = (y_pred_probs >= best_threshold).astype(int)

print("\n--- דוח תוצאות סופי ---")
print(classification_report(y_test, y_pred_tuned))

סף אופטימלי (Threshold) למקסום F1: 0.4398

--- דוח תוצאות סופי ---
              precision    recall  f1-score   support

           0       0.68      0.51      0.58       377
           1       0.93      0.97      0.95      2729

    accuracy                           0.91      3106
   macro avg       0.81      0.74      0.77      3106
weighted avg       0.90      0.91      0.91      3106

